# 🚀 Financial Data Pipeline: Ingestion, Database Storage & Vector Indexing

This notebook implements a clean **Three-Module Architecture (Pure In-Memory & Database Persistence)**:

### 1. 📥 Data Acquisition (`get_data.py` - Pure In-Memory)
- **Scrape Tickers**: Dynamic in-memory constituent scraping via Slickcharts (`get_data.get_slickcharts_ticker`).
- **Fetch Fundamentals**: Concurrently calculate financial ratios (ROIC, CAGR, Margins, FCF, P/E) via `yfinance` into an in-memory `List[Dict]` (`get_data.batch_fetch_fundamentals`).
- **Fetch News**: Concurrently extract market news headlines, summaries, and title-based deterministic UUIDs into an in-memory `Dict[str, List[Dict]]` (`get_data.batch_fetch_news`).
- **Zero Disk Writes**: No JSON files or temporary disk files used.

### 2. 💾 Database Persistence Layer (`sql_handler.py` & `chroma_handler.py`)
1. **Fundamentals SQLite DB (`sql_handler.py`)**: Persists metrics with atomic upserts directly into **`data/database/foundamental_sql.db`**.
2. **News & Fundamentals Vector DB (`chroma_handler.py`)**: Builds embeddings and indexes documents directly into **ChromaDB (`data/database/news_vector`)** with `doc_id`-based deduplication.

### 3. 🔄 Daily Orchestrator (`data_pipeline.py`)
- One-command end-to-end execution via `run_pipeline()`, runnable daily via CLI, cron, or Web 1-Click button.

---

## 0. Setup & Active Storage Status

In [ ]:
import sys
import os
import json
import time
import pandas as pd

# Ensure project root and local get_data are in sys.path
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

# Import the three dedicated modules and the orchestrator
from get_data import get_data, sql_handler, chroma_handler, data_pipeline

print("✓ get_data, sql_handler, and chroma_handler loaded successfully.")
print(f"  • Fundamentals SQLite DB (Active) : {sql_handler.DB_PATH} (Current: {sql_handler.count_fundamentals()} stocks)")
print(f"  • News Vector DB Chroma (Active)  : {chroma_handler.CHROMA_DIR} (Current: {chroma_handler.get_chroma_document_count()} indexed docs)")

## 🧪 Quick Sanity Check (Optional Test)
Tests fundamental calculation and news extraction on a single ticker (`AAPL`) in memory.

In [ ]:
test_ticker = "AAPL"
print(f"Testing single-stock analysis for {test_ticker}...")
test_fund = get_data.fundamental_analysis(test_ticker)
test_news_list = get_data.get_ticker_news(test_ticker, max_news=2)

if test_fund:
    print(f"✓ {test_fund['ticker']} ({test_fund['company_name']}) | Sector: {test_fund['sector']}")
    print(f"  Sample Metrics: P/E={test_fund['fundamentals'].get('pe_ratio')}, ROIC={test_fund['fundamentals'].get('roic')}")
print(f"✓ News articles fetched via get_data: {len(test_news_list)}")
if test_news_list:
    print(f"  Headline: {test_news_list[0].get('title')}")
    print(f"  doc_id  : {test_news_list[0].get('doc_id')}")

## ========================================================
## 📥 PROCESS 1: Getting the Data (`get_data.py` - Pure In-Memory)
## ========================================================
In this phase, we fetch and compute data over the network and keep it in Python RAM. **Zero database writes or JSON files.**

### 1.1 Scrape Constituent Tickers via Slickcharts

In [ ]:
# Fetch NASDAQ-100 tickers (set limit=100 for full index, or limit=5 for a quick run)
INDEX_LIMIT = 100
tickers = get_data.get_slickcharts_ticker(index_type="nasdaq100", limit=INDEX_LIMIT)

print(f"✓ Process 1: Scraped {len(tickers)} constituent tickers in-memory.")
print("Sample tickers:", tickers[:12])

### 1.2 Concurrently Fetch Fundamentals into Memory (`List[Dict]`)
Calculates balance sheet / cash flow ratios concurrently in RAM.

In [ ]:
# Target tickers: tickers (all 100) or a slice like tickers[:5] for quick testing
target_tickers = tickers

print(f"Fetching fundamentals in-memory for {len(target_tickers)} tickers...")
fund_start = time.time()

in_memory_fundamentals = get_data.batch_fetch_fundamentals(
    tickers=target_tickers,
    max_workers=6,
    batch_size=25,
    delay=1.0
)

fund_elapsed = round(time.time() - fund_start, 2)
print(f"\n✓ Process 1: Retrieved {len(in_memory_fundamentals)} fundamental records into memory ({fund_elapsed}s).")

### 1.3 Concurrently Fetch News into Memory via `get_data.py` (`Dict[str, List[Dict]]`)
Extracts latest news headlines, summaries, and UUIDs concurrently into RAM.

In [ ]:
print(f"Fetching news in-memory for {len(target_tickers)} tickers via get_data...")
news_start = time.time()

in_memory_news = get_data.batch_fetch_news(
    tickers=target_tickers,
    max_workers=8,
    max_news_per_ticker=10
)

news_elapsed = round(time.time() - news_start, 2)
total_in_memory_articles = sum(len(v) for v in in_memory_news.values())
print(f"\n✓ Process 1: Retrieved {total_in_memory_articles} news articles across {len(in_memory_news)} tickers ({news_elapsed}s).")

### 1.4 Inspect In-Memory Data Before Database Commit
Verify the extracted data in RAM before committing to SQLite and ChromaDB.

In [ ]:
if in_memory_fundamentals:
    sample_stock = in_memory_fundamentals[0]
    print(f"Sample In-Memory Stock: {sample_stock['ticker']} ({sample_stock['company_name']})")
    print("Key Ratios:", {
        k: sample_stock['fundamentals'].get(k)
        for k in ["pe_ratio", "roic", "gross_margin", "operating_margin", "revenue_growth_1y"]
    })

sample_ticker = target_tickers[0]
sample_articles = in_memory_news.get(sample_ticker, [])
print(f"\nSample In-Memory News for {sample_ticker} ({len(sample_articles)} articles):")
if sample_articles:
    print("  Title :", sample_articles[0].get("title"))
    print("  doc_id:", sample_articles[0].get("doc_id"))
    print("  Source:", sample_articles[0].get("publisher"))

## ========================================================
## 💾 PROCESS 2: Save Data into SQLite & ChromaDB
## ========================================================
Now we take the in-memory data structures from Process 1 and persist them directly into databases:
1. **SQLite Database (`sql_handler.py`)**: `foundamental_sql.db` for stock screening & metrics.
2. **ChromaDB Vector Store (`chroma_handler.py`)**: `news_vector` for semantic news search & headline queries with `doc_id` deduplication.

### 2.1 Save Fundamentals -> SQLite DB via `sql_handler.py`

In [ ]:
# Persist in-memory fundamentals into SQLite (foundamental_sql.db)
sql_handler.save_fundamentals(in_memory_fundamentals)

total_db = sql_handler.count_fundamentals()
print(f"✓ Process 2: Stored {len(in_memory_fundamentals)} records. Total in SQLite DB: {total_db}")

### 2.2 Inspect SQLite Database Records (`sql_handler.py`)

In [ ]:
# Query SQLite database and display in a pandas DataFrame
db_records = sql_handler.fetch_all_fundamentals()
df_db = pd.DataFrame([
    {
        "ticker": r.get("ticker"),
        "company_name": r.get("company_name"),
        "sector": r.get("sector"),
        **r.get("fundamentals", {})
    }
    for r in db_records
])

# cols = ["ticker", "company_name", "sector", "pe_ratio", "roic", "gross_margin", "operating_margin", "fcf_margin"]
df_db[[c for c in df_db.columns.tolist()]].head(10)

### 2.3 Index News & Fundamentals into ChromaDB (`chroma_handler.py`)
Indexes documents directly into the vector database using `doc_id` deduplication to prevent duplicate embeddings.

In [ ]:
print(f"Indexing into ChromaDB Vector Store ({chroma_handler.CHROMA_DIR}) via chroma_handler...")
vec_start = time.time()

# Index directly using in-memory data with doc_id deduplication
vector_index = chroma_handler.index_all_data(
    fundamentals_data=in_memory_fundamentals,
    news_data=in_memory_news,
)

vec_elapsed = round(time.time() - vec_start, 2)
current_chroma_count = chroma_handler.get_chroma_document_count()
print(f"\n✓ Process 2: ChromaDB vector indexing complete in {vec_elapsed}s.")
print(f"  Total operational documents in ChromaDB: {current_chroma_count}")

### 2.4 Multi-Database Verification Summary

In [ ]:
print("=== Database Storage Summary ===")
print(f"1. [Active] SQLite DB (foundamental_sql.db) : {sql_handler.count_fundamentals()} stocks indexed")
print(f"2. [Active] Chroma Vector DB (news_vector)  : {chroma_handler.get_chroma_document_count()} embedded documents")
print("================================")

## 🔄 Section 3: One-Click Daily Automation Pipeline Execution
Executes the entire workflow via `data_pipeline.run_pipeline()` in one step. Can be run daily via CLI or web.

In [ ]:
# Set sample_limit=None for all 100 tickers, or e.g. 5 for a quick run
summary = data_pipeline.run_pipeline(
    refresh_fundamentals=False, # Set to True to re-scrape all financial statements
    refresh_news=True,          # Set to True to fetch latest market news
)
print("\nPipeline Result:")
print(json.dumps(summary, indent=2))